# Prepare the shared FineWeb-Edu token corpus

Select a CPU runtime in Colab, run the cells in order, and approve the Google Drive mount when prompted.


In [ ]:
# Experiment storage
USE_GOOGLE_DRIVE = True

from pathlib import Path
import os

default_workspace = "/content/bmr" if Path("/content").is_dir() else "~/bmr-workspace"
WORKSPACE = Path(os.environ.get("BMR_WORKSPACE", default_workspace)).expanduser().resolve()
EXPERIMENT = "standard_vs_gated_160m_tinyllama"
DATA = WORKSPACE / "data" / "fineweb_edu_tinyllama"
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    my_drive = Path("/content/drive/MyDrive")
    if not my_drive.is_dir():
        raise RuntimeError("Mount Google Drive before continuing.")
    storage = my_drive / "bmr"
else:
    storage = WORKSPACE
RUNS = storage / "runs" / EXPERIMENT
SAVED_DATA = storage / "data" / "fineweb_edu_tinyllama"
print("Experiment outputs:", RUNS)


In [ ]:
# Set up experiment runtime
"""Standard-library setup embedded in the notebooks so it can run before the checkout exists."""

import fcntl
import json
import re
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version

GITHUB_REPO_URL = "https://github.com/seungjun-green/base-model-research"


def _checkout(workspace, runs):
    source_file = runs / "source.json"
    if source_file.exists():
        source = json.loads(source_file.read_text())
        if source["url"].rstrip("/").removesuffix(".git").lower() != GITHUB_REPO_URL.lower():
            raise ValueError(
                f"{runs} is already pinned to a different repository. Use a new EXPERIMENT name."
            )
        commit = source["commit"]
    else:
        head = subprocess.check_output(
            ["git", "ls-remote", "--exit-code", GITHUB_REPO_URL, "HEAD"], text=True
        )
        commit = head.split()[0]
        source = {"url": GITHUB_REPO_URL, "commit": commit}
    if not re.fullmatch(r"[0-9a-f]{40}", commit):
        raise ValueError(f"Invalid Git commit for this experiment: {commit!r}")

    repo = workspace / "repos" / commit
    if not repo.exists():
        repo.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(["git", "clone", "--", GITHUB_REPO_URL, repo], check=True)
    if subprocess.check_output(["git", "status", "--porcelain"], cwd=repo, text=True).strip():
        raise RuntimeError(f"The runtime checkout has local edits: {repo}")
    if subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repo, text=True).strip() != commit:
        subprocess.run(["git", "fetch", "origin", commit], cwd=repo, check=True)
        subprocess.run(["git", "checkout", "--detach", commit], cwd=repo, check=True)
    if not source_file.exists():
        temporary = source_file.with_suffix(".tmp")
        temporary.write_text(json.dumps(source, indent=2) + "\n")
        temporary.replace(source_file)
    return repo


def setup_colab(workspace, runs, *, benchmark):
    runs.mkdir(parents=True, exist_ok=True)
    with (runs / ".setup.lock").open("a") as lock:
        fcntl.flock(lock, fcntl.LOCK_EX)
        repo = _checkout(workspace, runs)
        try:
            torch_version = version("torch")
        except PackageNotFoundError as error:
            raise RuntimeError("Select a Colab runtime with PyTorch installed and rerun setup.") from error

        # Keep the runtime's PyTorch build, including its CUDA version suffix.
        extras = "data,eval" if benchmark else "data"
        print(f"Using {sys.executable} with PyTorch {torch_version}", flush=True)
        subprocess.run(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "--quiet",
                f"torch=={torch_version}",
                "-e",
                f"{repo}[{extras}]",
            ],
            check=True,
        )
    return repo

REPO = setup_colab(WORKSPACE, RUNS, benchmark=False)

def run(*arguments):
    subprocess.run([sys.executable, "-m", "base_model_research.cli", *map(str, arguments)],
                   cwd=REPO, check=True)


Prepare once and reuse for every run. Default: 2,048 held-out whole documents, followed by 3,167,907,840 training targets plus one lookahead token. uint32 tokens occupy about 12.7 GB in decimal units, plus validation, tokenizer and caches. If preparation stops, rerun this cell. Tokenization starts over if needed. A partial Drive copy resumes from verified shards.


In [ ]:
import shutil
from base_model_research.config import DEFAULT_TOKENIZER, DEFAULT_TRAINING_TARGETS

print("Free disk GB:", shutil.disk_usage(WORKSPACE).free / 1e9)
prepared = SAVED_DATA if (SAVED_DATA / "manifest.json").is_file() else DATA
if not (prepared / "manifest.json").is_file():
    run("prepare-data", "--output", prepared)
manifest = json.loads((prepared / "manifest.json").read_text())
if (manifest["tokenizer"] != DEFAULT_TOKENIZER
        or manifest["splits"]["train"]["tokens"] < DEFAULT_TRAINING_TARGETS + 1):
    raise ValueError(f"Expected the full TinyLlama-tokenized experiment corpus at {prepared}.")
if prepared != SAVED_DATA:
    run("copy-data", "--source", prepared, "--destination", SAVED_DATA)
print("Prepared corpus:", SAVED_DATA)
